# Evaluate using AI as Judge Quality Evaluators with Azure AI Evaluation SDK

## Objective

This tutorial provides a step-by-step guide on how to evaluate prompts against variety of model endpoints deployed on Azure AI Platform or non Azure AI platforms. 

This guide uses Python Class as an application target which is passed to Evaluate API provided by Azure AI Evaluation SDK to evaluate results generated by LLM models against provided prompts. 

This tutorial uses the following Azure AI services:

- [azure-ai-evaluation](https://learn.microsoft.com/en-us/azure/ai-studio/how-to/develop/evaluate-sdk)

## Time

You should expect to spend 30 minutes running this sample. 

## About this example

This example demonstrates evaluating model endpoints responses against provided prompts using azure-ai-evaluation

## Before you begin

### Installation

Install the following packages required to execute this notebook. 

In [1]:
# %pip install azure-ai-evaluation
# %pip install promptflow-azure
# %pip install azure-identity
# %pip install --upgrade openai

### Parameters and imports

In [2]:
from pprint import pprint
import pandas as pd
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv
load_dotenv()

True

## Target Application

We will use Evaluate API provided by Prompt Flow SDK. It requires a target Application or python Function, which handles a call to LLMs and retrieve responses. 

In the notebook, we will use an Application Target `ModelEndpoints` to get answers from multiple model endpoints against provided question aka prompts. 

This application target requires list of model endpoints and their authentication keys. For simplicity, we have provided them in the `env_var` variable which is passed into init() function of `ModelEndpoints`.


Please provide Azure AI Project details so that traces and eval results are pushing in the project in Azure AI Studio.

In [3]:
# azure_ai_project = {
#     "subscription_id": "<your-subscription-id>",
#     "resource_group_name": "<your-resource-group-name>",
#     "project_name": "<your-project-name>",
# }

# azure_openai_api_version = "<api version>"
# azure_openai_deployment = "<your-deployment>"
# azure_openai_endpoint = "<your-endpoint>"
# azure_openai_api_key

## Data

Following code reads Json file "data.jsonl" which contains inputs to the Application Target function. It provides question, context and ground truth on each line. 

In [4]:
df = pd.read_json("data.jsonl", lines=True)
print(df.head())

                                           query  \
0                 What is the capital of France?   
1             Which tent is the most waterproof?   
2           Which camping table is the lightest?   
3  How much does TrailWalker Hiking Shoes cost?    

                                             context  \
0                   France is the country in Europe.   
1  #TrailMaster X4 Tent, price $250,## BrandOutdo...   
2  #BaseCamp Folding Table, price $60,## BrandCam...   
3  #TrailWalker Hiking Shoes, price $110## BrandT...   

                                        ground_truth  
0                                              Paris  
1  The TrailMaster X4 tent has a rainfly waterpro...  
2  The BaseCamp Folding Table has a weight of 15 lbs  
3    The TrailWalker Hiking Shoes are priced at $110  


## Configuration
To use Relevance and Cohenrence Evaluator, we will Azure Open AI model details as a Judge that can be passed as model config.

In [5]:
import os
from azure.ai.evaluation import AzureOpenAIModelConfiguration

azure_ai_project = os.environ.get("AZURE_AI_PROJECT")

model_config = AzureOpenAIModelConfiguration(
    azure_deployment=os.environ.get("AZURE_OPENAI_DEPLOYMENT"),
    azure_endpoint=os.environ.get("AZURE_OPENAI_ENDPOINT"),
    api_version=os.environ.get("AZURE_OPENAI_API_VERSION"),
    # api_key=os.environ.get("AZURE_OPENAI_API_KEY"),
)

print(os.environ.get("AZURE_OPENAI_API_KEY"))
print(os.environ.get("AZURE_OPENAI_ENDPOINT"))
print(os.environ.get("AZURE_OPENAI_API_VERSION"))
print(os.environ.get("AZURE_OPENAI_DEPLOYMENT"))


1ecgTXR8OPaoeSSgW9Q3uC9yKwpmaNNkFFyHk69yKirSZPh7UacWJQQJ99BDAC8vTInXJ3w3AAAAACOGotJ7
https://build-2025-fdp-test-account1.services.ai.azure.com
2024-12-01-preview
gpt-4.1


In [6]:
from model_endpoint import ModelEndpoint

model = ModelEndpoint(model_config)
model(query="Test")

{'response': "Hello! It looks like you're testing the chat. How can I assist you today?"}

## Run the evaluation

The Following code runs Evaluate API and uses Content Safety, Relevance and Coherence Evaluator to evaluate results from different models.

The following are the few parameters required by Evaluate API. 

+   Data file (Prompts): It represents data file 'data.jsonl' in JSON format. Each line contains question, context and ground truth for evaluators.     

+   Application Target: It is name of python class which can route the calls to specific model endpoints using model name in conditional logic.  

+   Model Name: It is an identifier of model so that custom code in the App Target class can identify the model type and call respective LLM model using endpoint URL and auth key.  

+   Evaluators: List of evaluators is provided, to evaluate given prompts (questions) as input and output (answers) from LLM models. 

In [7]:
import pathlib

from azure.ai.evaluation import evaluate
from azure.ai.evaluation import (
    ContentSafetyEvaluator,
    RelevanceEvaluator,
    CoherenceEvaluator,
    GroundednessEvaluator,
    FluencyEvaluator,
    SimilarityEvaluator,
)
from model_endpoint import ModelEndpoint


content_safety_evaluator = ContentSafetyEvaluator(
    azure_ai_project=azure_ai_project, credential=DefaultAzureCredential()
)
relevance_evaluator = RelevanceEvaluator(model_config)
coherence_evaluator = CoherenceEvaluator(model_config)
groundedness_evaluator = GroundednessEvaluator(model_config)
fluency_evaluator = FluencyEvaluator(model_config)
similarity_evaluator = SimilarityEvaluator(model_config)

path = str(pathlib.Path(pathlib.Path.cwd())) + "/data.jsonl"

results = evaluate(
    evaluation_name="Eval-Run-" + "-" + model_config["azure_deployment"].title(),
    data=path,
    target=ModelEndpoint(model_config),
    evaluators={
        "content_safety": content_safety_evaluator,
        "coherence": coherence_evaluator,
        "relevance": relevance_evaluator,
        "groundedness": groundedness_evaluator,
        "fluency": fluency_evaluator,
        "similarity": similarity_evaluator,
    },
    evaluator_config={
        "content_safety": {"column_mapping": {"query": "${data.query}", "response": "${target.response}"}},
        "coherence": {"column_mapping": {"response": "${target.response}", "query": "${data.query}"}},
        "relevance": {
            "column_mapping": {"response": "${target.response}", "context": "${data.context}", "query": "${data.query}"}
        },
        "groundedness": {
            "column_mapping": {
                "response": "${target.response}",
                "context": "${data.context}",
                "query": "${data.query}",
            }
        },
        "fluency": {
            "column_mapping": {"response": "${target.response}", "context": "${data.context}", "query": "${data.query}"}
        },
        "similarity": {
            "column_mapping": {"response": "${target.response}", "context": "${data.context}", "query": "${data.query}"}
        },
    },
)

Class ContentSafetyEvaluator: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class ViolenceEvaluator: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class SexualEvaluator: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class SelfHarmEvaluator: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class HateUnfairnessEvaluator: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
[2025-05-01 23:39:15 -0700][promptflow._sdk._orchestrator.run_submitter][INFO] - Submitting run azure_ai_evaluation_evaluators_TARGET_20250501_233901_680421, log path: C:\Users\anksing\.promptflow\.runs\azure_ai_evaluation_evaluato

2025-05-01 23:39:16 -0700   20716 execution.bulk     INFO     Current thread is not main thread, skip signal handler registration in BatchEngine.
2025-05-01 23:39:16 -0700   20716 execution          WARNING  Starting run without column mapping may lead to unexpected results. Please consult the following documentation for more information: https://aka.ms/pf/column-mapping
2025-05-01 23:39:16 -0700   20716 execution.bulk     INFO     Current system's available memory is 28092.6484375MB, memory consumption of current process is 303.73828125MB, estimated available worker count is 28092.6484375/303.73828125 = 92
2025-05-01 23:39:16 -0700   20716 execution.bulk     INFO     Set process count to 4 by taking the minimum value among the factors of {'default_worker_count': 4, 'row_count': 4, 'estimated_worker_count_based_on_memory_usage': 92}.
2025-05-01 23:39:24 -0700   20716 execution.bulk     INFO     Process name(SpawnProcess-4)-Process id(47656)-Line number(0) start execution.
2025-05-01 23

[2025-05-01 23:40:00 -0700][promptflow._core.entry_meta_generator][WARNING] - Generate meta in current process and timeout won't take effect. Please handle timeout manually outside current process.
[2025-05-01 23:40:00 -0700][promptflow._core.entry_meta_generator][WARNING] - Generate meta in current process and timeout won't take effect. Please handle timeout manually outside current process.
[2025-05-01 23:40:00 -0700][promptflow._core.entry_meta_generator][WARNING] - Generate meta in current process and timeout won't take effect. Please handle timeout manually outside current process.
[2025-05-01 23:40:00 -0700][promptflow._core.entry_meta_generator][WARNING] - Generate meta in current process and timeout won't take effect. Please handle timeout manually outside current process.
[2025-05-01 23:40:00 -0700][promptflow._core.entry_meta_generator][WARNING] - Generate meta in current process and timeout won't take effect. Please handle timeout manually outside current process.
[2025-05-0

2025-05-01 23:40:00 -0700   20716 execution.bulk     INFO     Current thread is not main thread, skip signal handler registration in BatchEngine.
2025-05-01 23:40:00 -0700   20716 execution.bulk     INFO     Finished 3 / 3 lines.
2025-05-01 23:40:00 -0700   20716 execution.bulk     INFO     Average execution time for completed lines: 0.06 seconds. Estimated time for incomplete lines: 0.0 seconds.
2025-05-01 23:40:00 -0700   20716 execution          ERROR    3/3 flow run failed, indexes: [2,3,1], exception of index 2: (UserError) SimilarityEvaluator: Either 'conversation' or individual inputs must be provided.
======= Run Summary =======

Run name: "azure_ai_evaluation_evaluators_similarity_20250501_233959_845927"
Run status: "Completed"
Start time: "2025-05-01 23:39:59.910466-07:00"
Duration: "0:00:01.779287"
Output path: "C:\Users\anksing\.promptflow\.runs\azure_ai_evaluation_evaluators_similarity_20250501_233959_845927"



[2025-05-01 23:40:27 -0700][promptflow.core._prompty_utils][ERROR] - Exception occurs: CredentialUnavailableError: Failed to invoke the Azure CLI
[2025-05-01 23:40:27 -0700][promptflow.core._prompty_utils][ERROR] - Exception occurs: CredentialUnavailableError: Failed to invoke the Azure CLI
[2025-05-01 23:40:27 -0700][promptflow.core._prompty_utils][ERROR] - Exception occurs: CredentialUnavailableError: Failed to invoke the Azure CLI
[2025-05-01 23:40:28 -0700][promptflow.core._prompty_utils][ERROR] - Exception occurs: CredentialUnavailableError: Failed to invoke the Azure CLI
[2025-05-01 23:40:41 -0700][promptflow.core._prompty_utils][ERROR] - Exception occurs: CredentialUnavailableError: Failed to invoke the Azure CLI
[2025-05-01 23:40:41 -0700][promptflow.core._prompty_utils][ERROR] - Exception occurs: CredentialUnavailableError: Failed to invoke the Azure CLI
[2025-05-01 23:40:41 -0700][promptflow.core._prompty_utils][ERROR] - Exception occurs: CredentialUnavailableError: Failed to

...

2025-05-01 23:40:00 -0700   20716 execution.bulk     INFO     Current thread is not main thread, skip signal handler registration in BatchEngine.
2025-05-01 23:41:04 -0700   20716 execution.bulk     INFO     Finished 3 / 3 lines.
2025-05-01 23:41:04 -0700   20716 execution.bulk     INFO     Average execution time for completed lines: 21.23 seconds. Estimated time for incomplete lines: 0.0 seconds.
2025-05-01 23:41:04 -0700   20716 execution          ERROR    3/3 flow run failed, indexes: [2,3,1], exception of index 2: Failed to invoke the Azure CLI
======= Run Summary =======

Run name: "azure_ai_evaluation_evaluators_content_safety_20250501_233959_832691"
Run status: "Completed"
Start time: "2025-05-01 23:39:59.911947-07:00"
Duration: "0:01:04.603344"
Output path: "C:\Users\anksing\.promptflow\.runs\azure_ai_evaluation_evaluators_content_safety_20250501_233959_832691"



c:\Users\anksing\.conda\envs\azure-ai-evaluation-dev\Lib\site-packages\promptflow\_sdk\operations\_local_storage_operations.py:516: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '(Failed)' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  outputs.fillna(value="(Failed)", inplace=True)  # replace nan with explicit prompt
C:\Users\anksing\Repos\azure\azure-sdk-for-python\sdk\evaluation\azure-ai-evaluation\azure\ai\evaluation\_evaluate\_batch_run\proxy_client.py:81: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  result_df.replace("(Failed)", math.nan, inplace=True)


======= Combined Run Summary (Per Evaluator) =======

{
    "content_safety": {
        "status": "Completed with Errors",
        "duration": "0:01:04.603344",
        "completed_lines": 0,
        "failed_lines": 3,
        "log_path": "C:\\Users\\anksing\\.promptflow\\.runs\\azure_ai_evaluation_evaluators_content_safety_20250501_233959_832691"
    },
    "coherence": {
        "status": "Completed with Errors",
        "duration": "0:01:03.334193",
        "completed_lines": 0,
        "failed_lines": 3,
        "log_path": "C:\\Users\\anksing\\.promptflow\\.runs\\azure_ai_evaluation_evaluators_coherence_20250501_233959_832691"
    },
    "relevance": {
        "status": "Completed with Errors",
        "duration": "0:01:03.217215",
        "completed_lines": 0,
        "failed_lines": 3,
        "log_path": "C:\\Users\\anksing\\.promptflow\\.runs\\azure_ai_evaluation_evaluators_relevance_20250501_233959_832691"
    },
    "groundedness": {
        "status": "Completed with Errors",

c:\Users\anksing\.conda\envs\azure-ai-evaluation-dev\Lib\site-packages\promptflow\_sdk\operations\_local_storage_operations.py:516: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '(Failed)' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  outputs.fillna(value="(Failed)", inplace=True)  # replace nan with explicit prompt
C:\Users\anksing\Repos\azure\azure-sdk-for-python\sdk\evaluation\azure-ai-evaluation\azure\ai\evaluation\_evaluate\_batch_run\proxy_client.py:81: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  result_df.replace("(Failed)", math.nan, inplace=True)


{'metrics': {'fluency.binary_aggregate': 0.33,
             'fluency.fluency': 5.0,
             'fluency.fluency_threshold': 3.0,
             'fluency.gpt_fluency': 5.0,
             'groundedness.binary_aggregate': 0.0,
             'groundedness.gpt_groundedness': 1.0,
             'groundedness.groundedness': 1.0,
             'groundedness.groundedness_threshold': 3.0},
 'rows': [{'inputs.context': 'France is the country in Europe.',
           'inputs.ground_truth': 'Paris',
           'inputs.query': 'What is the capital of France?',
           'outputs.fluency.fluency': 5.0,
           'outputs.fluency.fluency_reason': 'The response demonstrates a high '
                                             'level of fluency, with '
                                             'sophisticated vocabulary, '
                                             'complex sentence structures, and '
                                             'excellent coherence. It is '
                           

View the results

In [8]:
pprint(results)

In [9]:
pd.DataFrame(results["rows"])

,outputs.response,inputs.query,inputs.context,inputs.ground_truth,outputs.groundedness.groundedness,outputs.groundedness.gpt_groundedness,outputs.groundedness.groundedness_reason,outputs.groundedness.groundedness_result,outputs.groundedness.groundedness_threshold,outputs.fluency.fluency,outputs.fluency.gpt_fluency,outputs.fluency.fluency_reason,outputs.fluency.fluency_result,outputs.fluency.fluency_threshold
0,NaN,What is the capital of France?,France is the country in Europe.,Paris,1.0,1.0,"The RESPONSE is entirely ungrounded, as it doe...",fail,3.0,5.0,5.0,The response demonstrates a high level of flue...,pass,3.0
1,The **most waterproof tent** depends on what y...,Which tent is the most waterproof?,"#TrailMaster X4 Tent, price $250,## BrandOutdo...",The TrailMaster X4 tent has a rainfly waterpro...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,The **lightest camping tables** are typically ...,Which camping table is the lightest?,"#BaseCamp Folding Table, price $60,## BrandCam...",The BaseCamp Folding Table has a weight of 15 lbs,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,"As of my latest update, **TrailWalker Hiking S...",How much does TrailWalker Hiking Shoes cost?,"#TrailWalker Hiking Shoes, price $110## BrandT...",The TrailWalker Hiking Shoes are priced at $110,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
